## Setup

This notebook requires the `.venv` virtual environment.

**Visual Studio Code** — select the kernel via the kernel picker (top-right corner of the notebook): choose `Python Environments… → .venv`.

**Terminal** — activate the environment manually, then launch Jupyter:

```bash
# Windows
.venv\Scripts\activate

# macOS / Linux
source .venv/bin/activate

jupyter notebook 002-prompt-evaluation.ipynb
```

In [2]:
# Install dependencies with uv (faster alternative to pip)
!uv pip install anthropic python-dotenv
%load_ext autoreload
%autoreload 2

Using Python 3.13.5 environment at: c:\Drives\T\Trainings\AI\Claude with Anthropic API\.venv
Audited 2 packages in 13ms


In [3]:
#Load environment variables from .env file
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
import json
import anthropic_client
from anthropic_client import AnthropicClient
from anthropic_message import AssistantMessage, UserMessage
from prompt_runner import PromptRunner

print(f"Loaded from: {anthropic_client.__file__}")
print(f"AnthropicClient version: {AnthropicClient.version}")

client = AnthropicClient.create()

Loaded from: c:\Drives\T\Trainings\AI\Claude with Anthropic API\python\anthropic_client.py
AnthropicClient version: 2026-04-16 12:02:39


In [5]:
# Generate test cases

def generate_dataset():
    prompt = """
    Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
    that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
    each representing task that requires Python, JSON, or a Regex to complete.

    Example output:
    ```json
    [
        {
            "task": "Description of task",
            "format": "json" or "python" or "regex",
            "solution_criteria": "Key criteria for evaluating the solution"
        },
        ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
    * Focus on tasks that do not require writing much code

    Please generate 3 objects.
    """
    generator = AnthropicClient.create()
    generator.append_message(UserMessage(prompt))
    generator.append_message(AssistantMessage("Sure! Here's a dataset of 3 tasks that require Python, JSON, or Regex for AWS-related tasks:\n```json"))
    response = generator.get_response(["```"])
    return json.loads(response)

# Generate the dataset and write it to 'dataset.json'
dataset = generate_dataset()
with open('dataset.json', 'w') as f:
    json.dump(dataset, f, indent=4)
print("Dataset generated and saved to 'dataset.json'")

Sending 2 messages to Claude
Context of (2 messages):
--------------------------------------------------
User

        Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
        that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
        each representing task that requires Python, JSON, or a Regex to complete.

        Example output:
        ```json
        [
            {
                "task": "Description of task",
                "format": "json" or "python" or "regex",
                "solution_criteria": "Key criteria for evaluating the solution"
            },
            ...additional
        ]
        ```

        * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular
    expression.
        * Focus on tasks that do not require writing much code

        Please generate 3 objects.

Assistant
    Sure! Here's a dataset 

In [6]:
# Prompt evaluation example — Prompt templates & dataset
from pathlib import Path

prompts = [
    """
# v2.0
Please answer the user's question:
{task}

* Respond only with Python, JSON, or a plain regex.
* Do not add any explanatory text or formatting, just the raw solution.
""",
]

prompt_template = prompts[0]

## Evaluation dataset
_notebook_dir = Path(globals().get("__vsc_ipynb_file__", Path.cwd())).parent
with open(_notebook_dir / "dataset.json") as f:
    dataset = json.load(f)
print(f"Loaded evaluation dataset with {len(dataset)} entries from dataset.json")

Loaded evaluation dataset with 3 entries from dataset.json


In [7]:
# Prompt evaluation example — Pass through Claude

runner = PromptRunner(dataset, client)
results = runner.runEval(prompt_template)

print("Evaluation results:")
print(json.dumps(results, indent=2))

Sending 2 messages to Claude
Context of (2 messages):
--------------------------------------------------
User

    # v2.0
    Please answer the user's question:
    Extract the AWS region from an S3 bucket ARN (e.g., 'arn:aws:s3:::my-bucket-us-east-1'). The region may or may
    not be present in the ARN.

    * Respond only with Python, JSON, or a plain regex.
    * Do not add any explanatory text or formatting, just the raw solution.
Assistant
    ```regex
--------------------------------------------------
API request successful

    (?:arn:aws:s3:::[\w\-]+-(us-east-1|us-east-2|us-west-1|us-west-2|eu-west-1|eu-central-1|ap-southeast-1|ap-
    southeast-2|ap-northeast-1|ca-central-1|sa-east-1|ap-south-1|eu-west-2|eu-west-3|eu-north-1|ap-northeast-2|ap-
    northeast-3|ap-east-1|me-south-1|af-south-1|eu-south-1)(?:[/-]|$)|arn:aws:s3:::[^-]*$)
Sending 2 messages to Claude
Context of (2 messages):
--------------------------------------------------
User

            You are an expert code

In [8]:
# Prompt evaluation example — Grade results
average_score = sum(r["score"]["score"] for r in results) / len(results)
print(f"Average score: {average_score:.2f}")

Average score: 8.25
